<a href="https://colab.research.google.com/github/e23378-Tharz/Statistical-Learning-e23378/blob/main/data_wrangling_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4: Data Wrangling




# what is Data Wrangling?

**Data Wrangling** (also called *data munging*) is the process of **cleaning, transforming, and preparing raw data** so it can be used for analysis or machine learning.



## Task 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

## Task 2: Load a Dataset

We use the **Titanic dataset** — a classic dataset containing information about passengers on the Titanic. It is commonly used for data wrangling practice because it has missing values, mixed types, and multiple columns to work with.

In [ ]:
# Load the Titanic dataset from a public URL
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns")

## Task 3: Explore the Dataset

Before doing anything else, we always look at the data first — this is called **Exploratory Data Analysis (EDA)**.

In [ ]:
# Show the first 5 rows
df.head()

In [ ]:
# Show column names, data types, and non-null counts
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

In [ ]:
# List all column names
print("Columns:", list(df.columns))

## Task 4: Handle Missing Values

Missing values are cells with no data. In pandas, they appear as `NaN` (Not a Number).  
We need to decide: **drop** them or **fill** them?

In [ ]:
# Count missing values in each column
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

In [ ]:
# Show as percentage
missing_pct = (df.isnull().sum() / len(df)) * 100
print("\nMissing value percentage:")
print(missing_pct[missing_pct > 0].round(2))

In [ ]:
# Fill missing 'Age' with the median age (median is better than mean for skewed data)
median_age = df['Age'].median()
df['Age'].fillna(median_age, inplace=True)

print(f"Filled missing Age values with median: {median_age}")

In [ ]:
# Fill missing 'Embarked' with the most common value (mode)
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'].fillna(mode_embarked, inplace=True)

print(f"Filled missing Embarked values with mode: {mode_embarked}")

In [ ]:
# Drop 'Cabin' column — too many missing values (>77%), not useful
df.drop(columns=['Cabin'], inplace=True)

print("Dropped 'Cabin' column.")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

## Task 5: Select and Filter Data

We can **select specific columns** or **filter rows** based on conditions.

In [ ]:
# Select only useful columns
df_clean = df[['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
print("Selected columns:", list(df_clean.columns))
df_clean.head()

In [ ]:
# Filter: only passengers who survived
survivors = df_clean[df_clean['Survived'] == 1]
print(f"Number of survivors: {len(survivors)} out of {len(df_clean)} total passengers")

In [ ]:
# Filter: female passengers in 1st class
female_first_class = df_clean[(df_clean['Sex'] == 'female') & (df_clean['Pclass'] == 1)]
print(f"Female passengers in 1st class: {len(female_first_class)}")
female_first_class.head()

## Task 6: Create New Columns (Feature Engineering)

We can create new columns by transforming existing ones. This is called **feature engineering**.

In [ ]:
# Create a 'FamilySize' column = siblings/spouses + parents/children + 1 (the passenger themselves)
df_clean = df_clean.copy()  # avoid SettingWithCopyWarning
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1

print("Family size distribution:")
print(df_clean['FamilySize'].value_counts().sort_index())

In [ ]:
# Create 'IsAlone' column: 1 if travelling alone, 0 if with family
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)

print(f"Passengers travelling alone: {df_clean['IsAlone'].sum()}")
print(f"Passengers with family: {(df_clean['IsAlone'] == 0).sum()}")

In [ ]:
# Create 'AgeGroup' column using pd.cut()
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins=[0, 12, 18, 60, 100],
    labels=['Child', 'Teen', 'Adult', 'Senior']
)

print("Age group distribution:")
print(df_clean['AgeGroup'].value_counts())

## Task 7: Grouping and Aggregation

`groupby()` lets us split data into groups and compute summary statistics for each group.

In [ ]:
# Average survival rate by passenger class
survival_by_class = df_clean.groupby('Pclass')['Survived'].mean().round(3) * 100
print("Survival rate (%) by passenger class:")
print(survival_by_class)

In [ ]:
# Average survival rate by sex
survival_by_sex = df_clean.groupby('Sex')['Survived'].mean().round(3) * 100
print("Survival rate (%) by sex:")
print(survival_by_sex)

In [ ]:
# Average age and fare by passenger class
class_summary = df_clean.groupby('Pclass').agg(
    avg_age=('Age', 'mean'),
    avg_fare=('Fare', 'mean'),
    count=('PassengerId', 'count')
).round(2)

print("Summary statistics by passenger class:")
print(class_summary)

## Task 8: Sorting and Renaming

In [ ]:
# Sort passengers by Fare (highest first)
top_fares = df_clean[['Name', 'Pclass', 'Fare', 'Survived']].sort_values('Fare', ascending=False).head(10)
print("Top 10 highest paying passengers:")
top_fares

In [ ]:
# Rename columns to be more descriptive
df_clean = df_clean.rename(columns={
    'SibSp': 'Siblings_Spouses',
    'Parch': 'Parents_Children',
    'Pclass': 'PassengerClass'
})

print("Updated column names:")
print(list(df_clean.columns))

## Task 9: Encoding Categorical Variables

Machine learning models need numbers, not text. So we convert text categories to numbers.

In [ ]:
# Map 'Sex' to numeric: male=0, female=1
df_clean['Sex_encoded'] = df_clean['Sex'].map({'male': 0, 'female': 1})

# Map 'Embarked' to numeric: S=0, C=1, Q=2
df_clean['Embarked_encoded'] = df_clean['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print("Encoding done.")
df_clean[['Sex', 'Sex_encoded', 'Embarked', 'Embarked_encoded']].head()

## Task 10: Simple Visualization of Wrangled Data

After wrangling, we always visualize to confirm our understanding.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Survival by Sex
df_clean.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'], edgecolor='black')
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival Rate')
axes[0].set_xticklabels(['Female', 'Male'], rotation=0)
axes[0].set_ylim(0, 1)

# Plot 2: Survival by Passenger Class
df_clean.groupby('PassengerClass')['Survived'].mean().plot(kind='bar', ax=axes[1], color='teal', edgecolor='black')
axes[1].set_title('Survival Rate by Passenger Class')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], rotation=0)
axes[1].set_ylim(0, 1)

# Plot 3: Age distribution
df_clean['Age'].hist(ax=axes[2], bins=20, color='purple', edgecolor='black', alpha=0.7)
axes[2].set_title('Age Distribution of Passengers')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('data_wrangling_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print("Plots saved.")

## Task 11: Save the Cleaned Dataset

In [ ]:
# Save the cleaned dataset to a CSV file
df_clean.to_csv('titanic_cleaned.csv', index=False)
print("Cleaned dataset saved as 'titanic_cleaned.csv'")
print(f"Final shape: {df_clean.shape}")
df_clean.head()